# Colab 03 - Evaluation / Ablation

Run final metrics after Qdrant Cloud and OpenRouter are configured.

Required Colab Secrets:
- `OPENROUTER_API_KEY`
- `QDRANT_URL`
- `QDRANT_API_KEY`


In [ ]:
REPO_URL = "https://github.com/phamdinhhai/project-ks2.git"
PROJECT_DIR = "/content/project-ks2"

import os
if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}
!git pull --ff-only


In [ ]:
!python -m pip install -U pip
!pip install -e ".[qdrant,agent,eval]"
!pip install requests


In [ ]:
# Configure secrets from Colab Secrets. Do not hardcode keys in notebook.
import os
try:
    from google.colab import userdata
    for name in ["OPENROUTER_API_KEY", "QDRANT_URL", "QDRANT_API_KEY"]:
        value = userdata.get(name)
        if value:
            os.environ[name] = value
except Exception as exc:
    print("Colab userdata unavailable:", exc)

os.environ.setdefault("OPENROUTER_MODEL", "google/gemini-2.5-flash")
os.environ.setdefault("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")

print("OPENROUTER_API_KEY set:", bool(os.environ.get("OPENROUTER_API_KEY")))
print("OPENROUTER_MODEL:", os.environ.get("OPENROUTER_MODEL"))
print("QDRANT_URL set:", bool(os.environ.get("QDRANT_URL")))
print("QDRANT_API_KEY set:", bool(os.environ.get("QDRANT_API_KEY")))


In [ ]:
# Provider diagnostics
!python -m medical_rag test-openrouter
!python -m medical_rag test-qdrant --qdrant-url "$QDRANT_URL" --use-cloud-auth


In [ ]:
# Baseline advanced metrics
!python scripts/colab_workflow.py eval-baseline   --eval-file data/eval_cases.json   --output-file outputs/benchmark/baseline_advanced.json


In [ ]:
# Agent metrics with Qdrant Cloud + OpenRouter
!python scripts/colab_workflow.py eval-agent   --eval-file data/eval_cases.json   --output-file outputs/benchmark/agent_openrouter.json   --use-qdrant


In [ ]:
# Ablation report
!python scripts/colab_workflow.py ablation   --eval-file data/eval_cases.json   --output-dir outputs/ablation


In [ ]:
# Save outputs to Drive if mounted
import os
if os.path.exists('/content/drive/MyDrive'):
    !mkdir -p /content/drive/MyDrive/medical_rag_outputs
    !cp -r outputs /content/drive/MyDrive/medical_rag_outputs/
else:
    print('Drive not mounted; outputs remain in /content/project-ks2/outputs')
